In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Основні датасети
transfers = pd.read_csv('data/raw/transfers.csv')
appearances = pd.read_csv('data/raw/appearances.csv')
players = pd.read_csv('data/raw/players.csv')
clubs = pd.read_csv('data/raw/clubs.csv')
competitions = pd.read_csv('data/raw/competitions.csv')

In [ ]:
#Опціональні для Feature Engineering
player_valuations = pd.read_csv('data/raw/player_valuations.csv')
national_teams = pd.read_csv('data/raw/national_teams.csv')
countries = pd.read_csv('data/raw/countries.csv')

In [29]:
def get_data_info(df):
    print(f"Shape: {df.shape}")
    print(f"Info: {df.info()}")
    print(f"Missing values:\n{df.isnull().sum()}")

In [21]:
#oбираємо потрібін атрибути 

transfers = transfers[[
    'player_id',
    'transfer_date',
    'transfer_season',
    'from_club_id',
    'to_club_id',
    'transfer_fee',
    'market_value_in_eur'
]]

players = players[[
    'player_id',
    'country_of_citizenship',
    'date_of_birth',
    'position',
    'sub_position',
    'foot',
    'height_in_cm'
]]

appearances = appearances[[
    'player_id',
    'player_club_id',
    'date',
    'competition_id',
    'yellow_cards',
    'red_cards',
    'goals',
    'assists',
    'minutes_played'
]]

clubs = clubs[[
    'club_id',
    'domestic_competition_id',
    'squad_size',
    'average_age',
    'foreigners_percentage',
    'national_team_players',
    'stadium_seats',
    'net_transfer_record'
]]

competitions = competitions[[
    'competition_id',
    'name',
    'sub_type',
    'type',
    'country_name',
    'confederation',
    'total_clubs'
]]


In [9]:
transfers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175165 entries, 0 to 175164
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   player_id            175165 non-null  int64  
 1   transfer_date        175165 non-null  object 
 2   transfer_season      175165 non-null  object 
 3   from_club_id         175165 non-null  int64  
 4   to_club_id           175165 non-null  int64  
 5   transfer_fee         113639 non-null  float64
 6   market_value_in_eur  107005 non-null  float64
dtypes: float64(2), int64(3), object(2)
memory usage: 9.4+ MB


In [11]:
transfers.head()

,player_id,transfer_date,transfer_season,from_club_id,to_club_id,transfer_fee,market_value_in_eur
0,467994,2030-06-30,25/26,5621,749,0.0,700000.0
1,645842,2028-02-02,27/28,6505,19684,0.0,150000.0
2,677470,2028-02-02,27/28,6505,3535,0.0,400000.0
3,709155,2028-02-02,27/28,6505,19684,0.0,325000.0
4,728133,2028-02-02,27/28,6505,35759,0.0,175000.0


In [12]:
transfers.isna().sum()

player_id                  0
transfer_date              0
transfer_season            0
from_club_id               0
to_club_id                 0
transfer_fee           61526
market_value_in_eur    68160
dtype: int64

In [13]:
transfers.value_counts()

player_id  transfer_date  transfer_season  from_club_id  to_club_id  transfer_fee  market_value_in_eur
2989       2006-07-01     06/07            27            41          2500000.0     3000000.0              1
517704     2018-01-31     17/18            10789         3323        0.0           25000.0                1
517705     2024-01-30     23/24            3345          2503        0.0           125000.0               1
           2023-08-08     23/24            2503          3345        0.0           150000.0               1
           2023-06-30     22/23            1701          2503        0.0           150000.0               1
                                                                                                         ..
271282     2019-08-27     19/20            16704         40812       1500000.0     1000000.0              1
           2019-06-30     18/19            170           16704       0.0           1000000.0              1
           2018-07-01     18/19  

In [16]:
# 1. Переводимо дату з тексту (object) у формат дати (datetime)
transfers['transfer_date'] = pd.to_datetime(transfers['transfer_date'])

# 2. Видаляємо повні дублікати, якщо вони є
transfers = transfers.drop_duplicates()

# 3. (прибираємо NaN та 0.0 — це оренди і вільні агенти)
transfers = transfers[transfers['transfer_fee'] > 0]

# 4. Прибираємо майбутні дати (датасет актуальний до 6 липня 2026 року)
transfers = transfers[transfers['transfer_date'] <= '2026-07-06']

# 5. Прибираємо аномалії, де клуб-продавець і клуб-покупець однакові
transfers = transfers[transfers['from_club_id'] != transfers['to_club_id']]

# 6. Оновлюємо індекси після фільтрації
transfers = transfers.reset_index(drop=True)

In [17]:
print("Новий розмір:", transfers.shape)
print("\nПропуски після очищення:")
print(transfers.isna().sum())
print("\nДіапазон дат:", transfers['transfer_date'].min(), "—", transfers['transfer_date'].max())

Новий розмір: (17547, 7)

Пропуски після очищення:
player_id                 0
transfer_date             0
transfer_season           0
from_club_id              0
to_club_id                0
transfer_fee              0
market_value_in_eur    1112
dtype: int64

Діапазон дат: 1997-07-01 00:00:00 — 2026-07-02 00:00:00


In [30]:
get_data_info(players)

Shape: (50149, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50149 entries, 0 to 50148
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   player_id               50149 non-null  int64  
 1   country_of_citizenship  49880 non-null  object 
 2   date_of_birth           50100 non-null  object 
 3   position                50149 non-null  object 
 4   sub_position            49563 non-null  object 
 5   foot                    44256 non-null  object 
 6   height_in_cm            45784 non-null  float64
dtypes: float64(1), int64(1), object(5)
memory usage: 2.7+ MB
Info: None
Missing values:
player_id                    0
country_of_citizenship     269
date_of_birth               49
position                     0
sub_position               586
foot                      5893
height_in_cm              4365
dtype: int64


In [31]:
players.head()

,player_id,country_of_citizenship,date_of_birth,position,sub_position,foot,height_in_cm
0,10,Germany,1978-06-09 00:00:00,Attack,Centre-Forward,right,184.0
1,26,Germany,1980-08-06 00:00:00,Goalkeeper,Goalkeeper,left,190.0
2,65,Bulgaria,1981-01-30 00:00:00,Attack,Centre-Forward,NaN,NaN
3,77,Brazil,1978-05-08 00:00:00,Defender,Centre-Back,NaN,NaN
4,80,Germany,1981-03-18 00:00:00,Goalkeeper,Goalkeeper,right,194.0


In [36]:
# 1. Переводимо дату народження у формат datetime
players['date_of_birth'] = pd.to_datetime(players['date_of_birth'])

# 2. Видаляємо можливі дублікати за player_id
players = players.drop_duplicates(subset=['player_id'])

# 3. Залишаємо лише гравців, які є в нашій очищеній таблиці transfers
players = players[players['player_id'].isin(transfers['player_id'])]

# 4. Видаляємо рядки без дати народження (без неї не порахувати вік гравця)
players = players.dropna(subset=['date_of_birth'])

# 5. Замінюємо пропуски в текстових колонках на 'Unknown'
players['country_of_citizenship'] = players['country_of_citizenship'].fillna('Unknown')
players['sub_position'] = players['sub_position'].fillna('Unknown')
players['foot'] = players['foot'].fillna('Unknown')

# 6. Перевіряємо зріст на аномалії (якщо < 140 см або > 220 см — ставимо NaN)
players.loc[(players['height_in_cm'] < 140) | (players['height_in_cm'] > 220), 'height_in_cm'] = np.nan

# 7. Оновлюємо індекси
players = players.reset_index(drop=True)




In [37]:
# 8. Перевіряємо результат
print("Новий розмір:", players.shape)
print("\nПропуски після очищення:")
print(players.isna().sum())
print("\nУнікальні позиції:")
print(players['position'].value_counts())

Новий розмір: (9949, 7)

Пропуски після очищення:
player_id                  0
country_of_citizenship     0
date_of_birth              0
position                   0
sub_position               0
foot                       0
height_in_cm              64
dtype: int64

Унікальні позиції:
position
Attack        3179
Defender      3162
Midfield      2833
Goalkeeper     774
Missing          1
Name: count, dtype: int64


In [ ]:
print("\nУнікальні значення foot:")
print(players['foot'].value_counts())
print("\nСтатистика зросту:")
print(players['height_in_cm'].describe())


Унікальні значення foot:
foot
right      6911
left       2533
both        306
Unknown     199
Name: count, dtype: int64

Статистика зросту:
count    9885.000000
mean      182.494082
std         6.980641
min       158.000000
25%       178.000000
50%       183.000000
75%       188.000000
max       206.000000
Name: height_in_cm, dtype: float64


In [40]:
#Видаляєм одного футболіста, у якого позиція Missing
players = players[players['position'] != 'Missing'].reset_index(drop=True)

In [41]:
get_data_info(appearances)

Shape: (1894350, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1894350 entries, 0 to 1894349
Data columns (total 9 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   player_id       int64 
 1   player_club_id  int64 
 2   date            object
 3   competition_id  object
 4   yellow_cards    int64 
 5   red_cards       int64 
 6   goals           int64 
 7   assists         int64 
 8   minutes_played  int64 
dtypes: int64(7), object(2)
memory usage: 130.1+ MB
Info: None
Missing values:
player_id         0
player_club_id    0
date              0
competition_id    0
yellow_cards      0
red_cards         0
goals             0
assists           0
minutes_played    0
dtype: int64


In [42]:
appearances.head()

,player_id,player_club_id,date,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,38004,853,2012-07-03,CLQ,0,0,2,0,90
1,79232,8841,2012-07-05,ELQ,0,0,0,0,90
2,42792,6251,2012-07-05,ELQ,0,0,0,0,45
3,73333,1274,2012-07-05,ELQ,0,0,0,0,90
4,122011,195,2012-07-05,ELQ,0,0,0,1,90


In [43]:
# 1. Переводимо дату у формат datetime
appearances['date'] = pd.to_datetime(appearances['date'])

# 2. Залишаємо статистику лише для гравців, які є в нашій очищеній таблиці players
appearances = appearances[appearances['player_id'].isin(players['player_id'])]

# 3. Видаляємо можливі повні дублікати
appearances = appearances.drop_duplicates()

# 4. Прибираємо можливі аномалії (матчі з 0 хвилин або від'ємні значення)
appearances = appearances[
    (appearances['minutes_played'] > 0) &
    (appearances['goals'] >= 0) &
    (appearances['assists'] >= 0) &
    (appearances['yellow_cards'] >= 0) &
    (appearances['red_cards'] >= 0)
]

# 5. Оновлюємо індекси
appearances = appearances.reset_index(drop=True)

In [44]:
# 6. Перевіряємо результат
print("Новий розмір:", appearances.shape)
print("\nДіапазон дат:", appearances['date'].min(), "—", appearances['date'].max())
print("\nСтатистика числових колонок:")
print(appearances[['yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played']].describe())

Новий розмір: (912653, 9)

Діапазон дат: 2012-07-05 00:00:00 — 2026-06-28 00:00:00

Статистика числових колонок:
        yellow_cards      red_cards          goals        assists  \
count  912653.000000  912653.000000  912653.000000  912653.000000   
mean        0.137563       0.003073       0.110971       0.084603   
std         0.353009       0.055354       0.356002       0.302151   
min         0.000000       0.000000       0.000000       0.000000   
25%         0.000000       0.000000       0.000000       0.000000   
50%         0.000000       0.000000       0.000000       0.000000   
75%         0.000000       0.000000       0.000000       0.000000   
max         2.000000       1.000000       6.000000       5.000000   

       minutes_played  
count   912653.000000  
mean        68.555736  
std         29.905315  
min          1.000000  
25%         45.000000  
50%         90.000000  
75%         90.000000  
max        148.000000  


In [45]:
get_data_info(clubs)

Shape: (796, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 796 entries, 0 to 795
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   club_id                  796 non-null    int64  
 1   domestic_competition_id  796 non-null    object 
 2   squad_size               796 non-null    int64  
 3   average_age              758 non-null    float64
 4   foreigners_percentage    736 non-null    float64
 5   national_team_players    796 non-null    int64  
 6   stadium_seats            796 non-null    int64  
 7   net_transfer_record      796 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 49.9+ KB
Info: None
Missing values:
club_id                     0
domestic_competition_id     0
squad_size                  0
average_age                38
foreigners_percentage      60
national_team_players       0
stadium_seats               0
net_transfer_record         0
dtype: int

In [46]:
clubs.head()

,club_id,domestic_competition_id,squad_size,average_age,foreigners_percentage,national_team_players,stadium_seats,net_transfer_record
0,10,L1,27,25.3,55.6,4,26515,+€5.90m
1,10004,FR1,27,27.2,51.9,7,19904,€-25.00m
2,10010,BRA1,32,26.6,28.1,3,47364,+€8.14m
3,1003,GB1,29,25.9,58.6,9,32259,+€57.30m
4,1005,IT1,39,23.8,92.3,10,31559,+€3.35m


In [47]:
# 1. Видаляємо можливі дублікати за club_id
clubs = clubs.drop_duplicates(subset=['club_id'])

# 2. Залишаємо лише клуби, які фігурують у нашій таблиці transfers (як продавець або покупець)
clubs = clubs[
    clubs['club_id'].isin(transfers['from_club_id']) |
    clubs['club_id'].isin(transfers['to_club_id'])
]

# 3. Проста функція для переведення net_transfer_record з тексту в число (в євро)
def parse_money(val):
    val = str(val).strip()
    if val in ['+-0', '0', '']:
        return 0.0
    
    # Прибираємо знаки '€' та '+' (знак '-' залишиться)
    val = val.replace('€', '').replace('+', '')
    
    if val.endswith('m'):
        return float(val[:-1]) * 1_000_000
    elif val.endswith('k'):
        return float(val[:-1]) * 1_000
    else:
        try:
            return float(val)
        except ValueError:
            return np.nan

clubs['net_transfer_record'] = clubs['net_transfer_record'].apply(parse_money)

# 4. Замінюємо 0 на NaN у колонках, де нуль означає відсутність даних
clubs.loc[clubs['squad_size'] == 0, 'squad_size'] = np.nan
clubs.loc[clubs['stadium_seats'] == 0, 'stadium_seats'] = np.nan

# 5. Оновлюємо індекси
clubs = clubs.reset_index(drop=True)

In [48]:
# 6. Перевіряємо результат
print("Новий розмір:", clubs.shape)
print("\nПропуски після очищення:")
print(clubs.isna().sum())
print("\nСтатистика числових колонок:")
print(clubs.describe())

Новий розмір: (764, 8)

Пропуски після очищення:
club_id                     0
domestic_competition_id     0
squad_size                 32
average_age                29
foreigners_percentage      45
national_team_players       0
stadium_seats               1
net_transfer_record         0
dtype: int64

Статистика числових колонок:
             club_id  squad_size  average_age  foreigners_percentage  \
count     764.000000  732.000000   735.000000             719.000000   
mean     8389.155759   28.355191    25.481497              40.420584   
std     16168.901652    5.391594     2.272136              20.090537   
min         3.000000    3.000000     0.000000               2.400000   
25%       592.000000   26.000000    24.500000              24.250000   
50%      2141.000000   28.000000    25.600000              40.000000   
75%      6997.000000   31.000000    26.700000              55.200000   
max    114977.000000   50.000000    30.100000             100.000000   

       national_tea

In [49]:
clubs.loc[clubs['average_age'] == 0, 'average_age'] = np.nan

In [51]:
clubs.head()

,club_id,domestic_competition_id,squad_size,average_age,foreigners_percentage,national_team_players,stadium_seats,net_transfer_record
0,10,L1,27.0,25.3,55.6,4,26515.0,5900000.0
1,10004,FR1,27.0,27.2,51.9,7,19904.0,-25000000.0
2,10010,BRA1,32.0,26.6,28.1,3,47364.0,8140000.0
3,1003,GB1,29.0,25.9,58.6,9,32259.0,57300000.0
4,1005,IT1,39.0,23.8,92.3,10,31559.0,3350000.0


In [50]:
get_data_info(competitions)

Shape: (65, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65 entries, 0 to 64
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   competition_id  65 non-null     object 
 1   name            65 non-null     object 
 2   sub_type        65 non-null     object 
 3   type            65 non-null     object 
 4   country_name    53 non-null     object 
 5   confederation   65 non-null     object 
 6   total_clubs     50 non-null     float64
dtypes: float64(1), object(6)
memory usage: 3.7+ KB
Info: None
Missing values:
competition_id     0
name               0
sub_type           0
type               0
country_name      12
confederation      0
total_clubs       15
dtype: int64


In [52]:
competitions.head()

,competition_id,name,sub_type,type,country_name,confederation,total_clubs
0,A1,bundesliga,first_tier,domestic_league,Austria,europa,12.0
1,AFAC,afc-asian-cup,afc_asian_cup,national_team_competition,NaN,asien,NaN
2,AFCN,africa-cup-of-nations,africa_cup_of_nations,national_team_competition,NaN,afrika,NaN
3,ARG1,torneo-apertura,first_tier,domestic_league,Argentina,amerika,30.0
4,AUS1,a-league-men,first_tier,domestic_league,Australia,asien,12.0


In [53]:
# 1. Видаляємо можливі дублікати за competition_id
competitions = competitions.drop_duplicates(subset=['competition_id'])

# 2. Заповнюємо пропуски в country_name значенням 'International'
competitions['country_name'] = competitions['country_name'].fillna('International')

# 3. Оновлюємо індекси
competitions = competitions.reset_index(drop=True)

# 4. Перевіряємо результат
print("Розмір:", competitions.shape)
print("\nПропуски:")
print(competitions.isna().sum())
print("\nТипи турнірів:")
print(competitions['type'].value_counts())


Розмір: (65, 7)

Пропуски:
competition_id     0
name               0
sub_type           0
type               0
country_name       0
confederation      0
total_clubs       15
dtype: int64

Типи турнірів:
type
domestic_league              31
other                        16
domestic_cup                 10
national_team_competition     5
international_cup             3
Name: count, dtype: int64


In [54]:
# 1. Залишаємо трансфери з 2013-07-01 та лише для гравців, які є в очищеному players
transfers = transfers[
    (transfers['transfer_date'] >= '2013-07-01') &
    (transfers['player_id'].isin(players['player_id']))
].reset_index(drop=True)

# 2. Синхронізуємо players, appearances та clubs під фінальний transfers
players = players[players['player_id'].isin(transfers['player_id'])].reset_index(drop=True)
appearances = appearances[appearances['player_id'].isin(transfers['player_id'])].reset_index(drop=True)
clubs = clubs[
    clubs['club_id'].isin(transfers['from_club_id']) |
    clubs['club_id'].isin(transfers['to_club_id'])
].reset_index(drop=True)

# 3. Перевіряємо фінальні розміри та пропуски в transfers
print("transfers:", transfers.shape)
print("players:", players.shape)
print("appearances:", appearances.shape)
print("clubs:", clubs.shape)
print("competitions:", competitions.shape)
print("\nПропуски в transfers.market_value_in_eur:", transfers['market_value_in_eur'].isna().sum())




transfers: (16624, 7)
players: (9779, 7)
appearances: (883515, 9)
clubs: (755, 8)
competitions: (65, 7)

Пропуски в transfers.market_value_in_eur: 886


In [ ]:
#Пропуски в transfers.market_value_in_eur: 886
#Для моделі з оцінкою їх можна буде або просто дропнути перед навчанням, або за бажанням дозаповнити пізніше з таблиці player_valuations.csv

In [58]:
# 4. Зберігаємо очищені таблиці
transfers.to_csv('data/processed/clean_transfers.csv', index=False)
players.to_csv('data/processed/clean_players.csv', index=False)
appearances.to_csv('data/processed/clean_appearances.csv', index=False)
clubs.to_csv('data/processed/clean_clubs.csv', index=False)
competitions.to_csv('data/processed/clean_competitions.csv', index=False)